# CNN t1 quantizada (8 bits) no SAT-6 

Notebook enxuto contendo apenas o necessário para:
1. Carregar o dataset SAT-6 (já "pickled" pelo `0_pickle_dataset.ipynb`);
2. Definir e treinar a topologia **t1** com quantização de **8 bits** (pesos e ativações);
3. Testar/avaliar a acurácia do modelo;
4. Exportar o modelo para **QONNX**, formato de entrada aceito pelo **hls4ml** (ao contrário do FINN, não é necessário aplicar as transformações de streamlining/conversão para hardware — isso é feito pelo próprio hls4ml a partir do ONNX/QONNX).

In [4]:
import os
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from tqdm import tqdm
import onnx

from utils.utils import read_object
from cnn_base import t1_CNN
import hls4ml
from sklearn.metrics import accuracy_score

%matplotlib inline
torch.manual_seed(4)

def print_dict(d, indent=0):
    for key, value in d.items():
        print('  ' * indent + str(key), end='')
        if isinstance(value, dict):
            print()
            print_dict(value, indent + 1)
        else:
            print(':' + ' ' * (20 - len(key) - 2 * indent) + str(value))


## 1. Load the trained model


In [5]:
batch_size = 512
device = torch.device('cpu')
test_dataset_output_name = "./dataset/test_dataset.pkl"

dataset_test = read_object(test_dataset_output_name)
loader_test = DataLoader(dataset_test, batch_size, shuffle=False)

X_test = dataset_test.tensors[0].numpy().astype(np.float32)
y_test = dataset_test.tensors[1].numpy().astype(np.int32)

In [9]:
trained_model = t1_CNN()
trained_model.load_state_dict(torch.load('./pytorch_models/sat6-cnn-t1.pt', map_location=device))
trained_model.eval()

y_model = []

with torch.no_grad():
    for x,y in loader_test:
        x = x.to(device, dtype=torch.float32)
        y = y.to(device, dtype=torch.long)
        pred = trained_model(x)
        y_model.append(pred)
y_model = torch.cat(y_model, dim=0)

print(f"y_model shape: {y_model.shape}")


y_model shape: torch.Size([81000, 6])


In [6]:
onnx_model_path = "./hls4ml_onnxs/sat6-cnn-t1.onnx"
onnx_model = onnx.load(onnx_model_path)

config = hls4ml.utils.config_from_onnx_model(
    onnx_model,
    granularity='name',
    backend='Vitis',
)

print('-----------------------------------')
print_dict(config)
print('-----------------------------------')

hls_model = hls4ml.converters.convert_from_onnx_model(
    onnx_model,
    output_dir='./hls4ml_prjs/hls4ml_prj_base_t1',
    backend='Vitis',
    hls_config=config,
    part='xc7z020clg400-1',
)

hls_model.compile()

Output layers:  ['Add_0']
Input shape: [28, 28, 4]
Topology:
Layer name: Conv_0, layer type: Conv, current shape: [[1, 28, 28, 4], [20, 3, 3, 4], [20]]
Layer name: Relu_0, layer type: Activation, current shape: [[1, 14, 14, 20]]
Layer name: Conv_1, layer type: Conv, current shape: [[1, 14, 14, 20], [8, 1, 1, 20], [8]]
Layer name: Relu_1, layer type: Activation, current shape: [[1, 14, 14, 8]]
Layer name: flatten_4, layer type: Reshape, current shape: [[1, 14, 14, 8]]
Layer name: MatMul_0, layer type: MatMul, current shape: [[1, 1568], [1568, 6]]
Layer name: Add_0, layer type: Merge, current shape: [[1, 6], [6]]
-----------------------------------
Model
  Precision
    default:         fixed<16,6>
  ReuseFactor:       1
  Strategy:          Latency
  BramFactor:        1000000000
  TraceOutput:       False
LayerName
  global_in
    Trace:           False
    Precision
      result:        auto
  Conv_0_param1
    Trace:           False
    Precision
      result:        auto
  Conv_1_pa

In [7]:
y_hls = hls_model.predict(np.ascontiguousarray(X_test))

In [10]:
print("PyTorch Accuracy: {}".format(accuracy_score(y_test, np.argmax(y_model, axis=1))))
print("hls4ml Accuracy: {}".format(accuracy_score(y_test, np.argmax(y_hls, axis=1))))

PyTorch Accuracy: 0.9794074074074074
hls4ml Accuracy: 0.03514814814814815


In [ ]:
hls_model.build(csim=False, synth=True)